# Semantic-fleet et claudish : une passerelle, plusieurs modeles reels

**Navigation** : [Index](README.md) | [<< Routage direct entre fournisseurs](Semantic-fleet-MultiConnector-Routing.ipynb)

Ce carnet branche l'organe `Connectors.AI.MultiConnector` sur la passerelle **claudish**, compatible OpenAI : un seul endpoint et une seule cle donnent acces a plusieurs modeles. Trois types de prompts sont routes vers trois alias distincts, avec de vraies reponses et le nom du connecteur effectivement choisi par l'organe.

Le carnet precedent cablait OpenAI et OpenRouter separement. Ici, seule la fabrique de connecteurs change : la table de signatures, `MultiTextCompletion` et l'arbitrage restent ceux de `semantic-fleet`.

## Objectifs pedagogiques

1. Decouvrir les alias reellement servis par la passerelle, sans supposer leur disponibilite.
2. Construire un `NamedTextCompletion` par alias avec la fabrique native `ClaudishConnectorConfiguration`.
3. Mesurer la latence de chaque alias, puis lire la selection effective de l'organe sur trois problemes differents.
4. Comprendre deux pieges : un secondaire a `VettingLevel.None` n'entre pas dans l'arbitrage, et le comparateur pondere privilegie le cout/duree le plus faible parmi les candidats admis.

## Prerequis et frontiere de la demonstration

- SDK .NET 9, kernel `.net-csharp`, sous-module initialise : `git submodule update --init MyIA.AI.Notebooks/GenAI/SemanticKernel/semantic-fleet`.
- `CLAUDISH_BASE_URL` et `CLAUDISH_API_KEY` dans l'environnement du processus. **Aucune cle n'est ecrite dans ce carnet.** Sous Windows, `setx CLAUDISH_API_KEY "..."` puis rouvrir le kernel conserve la cle hors du fichier.
- La fabrique claudish appartient a la branche amont `feature/t6b-prompt-router`, PR **`semantic-fleet#110` encore ouverte** au moment de l'execution. Ce consommateur epingle son contenu dans le worktree de demonstration ; il ne pretend pas que la PR est fusionnee. Le bump du gitlink CoursIA suivra le merge amont.
- Le wrapper console `tools/multiroute` resout la fermeture NuGet de Semantic Kernel et du SDK OpenAI dans son propre contexte de chargement. Cela evite les erreurs d'isolation ALC d'un `#r` direct dans .NET Interactive.

In [3]:
using System.Diagnostics;
using System.IO;
using System.Text;

// Les secrets restent dans l'environnement : on ne mesure que leur presence.
var baseUrl = Environment.GetEnvironmentVariable("CLAUDISH_BASE_URL");
var apiKey = Environment.GetEnvironmentVariable("CLAUDISH_API_KEY");
Console.WriteLine($"CLAUDISH_BASE_URL configure : {!string.IsNullOrWhiteSpace(baseUrl)}");
Console.WriteLine($"CLAUDISH_API_KEY presente  : {!string.IsNullOrWhiteSpace(apiKey)}");

var project = Path.GetFullPath(Path.Combine("semantic-fleet", "tools", "multiroute", "src", "SemanticFleet.MultiRoute.csproj"));
var wrapperName = "SemanticFleet.MultiRoute" + (OperatingSystem.IsWindows() ? ".exe" : "");
var wrapper = Path.GetFullPath(Path.Combine("semantic-fleet", "tools", "multiroute", "src", "bin", "Debug", "net9.0", wrapperName));
Console.WriteLine($"Projet present : {File.Exists(project)}");

Process RunProcess(string fileName, string arguments) {
    var process = new Process {
        StartInfo = new ProcessStartInfo {
            FileName = fileName, Arguments = arguments,
            RedirectStandardOutput = true, RedirectStandardError = true,
            StandardOutputEncoding = Encoding.UTF8, StandardErrorEncoding = Encoding.UTF8,
            UseShellExecute = false, CreateNoWindow = true,
        },
    };
    process.Start();
    return process;
}

Process RunWrapper(string arguments) => RunProcess(wrapper, arguments);

void PrintResult(string label, Process process) {
    process.WaitForExit();
    Console.WriteLine($"{label} (rc={process.ExitCode})");
    var output = process.StandardOutput.ReadToEnd().Trim();
    var error = process.StandardError.ReadToEnd().Trim();
    if (output.Length > 0) Console.WriteLine(output);
    if (error.Length > 0) Console.WriteLine(error);
}

if (!File.Exists(wrapper)) {
    PrintResult("dotnet build", RunProcess("dotnet", $"build \"{project}\" --nologo"));
}
Console.WriteLine($"Wrapper pret : {File.Exists(wrapper)}");

CLAUDISH_BASE_URL configure : True


CLAUDISH_API_KEY presente  : True


Projet present : True


Wrapper pret : True


## 1. Decouvrir la passerelle avant de router

`GET /models` ne genere aucun texte : il donne l'inventaire servi a cet instant. Le routage ne doit pas reposer sur une liste apprise par coeur.

In [4]:
var discovery = RunWrapper("--list-routes");
discovery.WaitForExit();
var aliases = discovery.StandardOutput.ReadToEnd()
    .Split(new[] { '\r', '\n' }, StringSplitOptions.RemoveEmptyEntries)
    .Select(line => line.Trim()).Where(line => line.Length > 0).ToArray();
Console.WriteLine($"Inventaire claudish : {aliases.Length} alias (rc={discovery.ExitCode})");
foreach (var alias in aliases) Console.WriteLine($"- {alias}");
var discoveryError = discovery.StandardError.ReadToEnd().Trim();
if (discoveryError.Length > 0) Console.WriteLine(discoveryError);

Inventaire claudish : 35 alias (rc=0)


- nb-minimax


- gpt-6-sol


- gpt-5.6-sol


- glm-5.3


- glm-5.2


... (26 outputs left out)

- glm-5-2


- mistral-medium-latest


- mistral-large-latest


- codestral-latest


## 2. La fabrique native, pas une reimplementation

Le wrapper appelle `ClaudishConnectorConfiguration.CreateFromEnvironment(routes)`, qui produit une `MultiOpenAICompatibleConnectorConfiguration` : chaque entree porte le meme endpoint et la meme cle, mais un `ModelId` different. `CreateNamedCompletions()` fournit ensuite les connecteurs a `WithMultiConnectorCompletionService`.

Trois alias sont retenus pour la demonstration : `qwen3.6-35b-a3b`, `glm-5.3` et `gpt-6.1-sol`. Les couts sont des **niveaux relatifs declares par l'appelant**, pas une facturation exposee par la passerelle.

## 3. Mesurer les candidats sans routage

Un meme prompt court est envoye directement a chacun des trois alias. Cette mesure separe deux questions souvent confondues : la latence d'un modele et le temps total d'un prompt route, qui depend aussi de la longueur de sa reponse. Une seule mesure n'est pas un benchmark : elle sert a initialiser `AverageDuration` avec une observation reelle, a renouveler lorsque la flotte change.

In [5]:
PrintResult("Appels directs, un par alias", RunWrapper("--measure"));

Appels directs, un par alias (rc=0)


{"alias":"qwen3.6-35b-a3b","latency_ms":3966,"chars":37,"preview":"\n\nLa capitale de la France est Paris."}
{"alias":"glm-5.3","latency_ms":1708,"chars":35,"preview":"La capitale de la France est Paris."}
{"alias":"gpt-6.1-sol","latency_ms":2187,"chars":35,"preview":"La capitale de la France est Paris."}


## 4. La table de routage

| Signature | Candidats admis | Resultat attendu de l'arbitrage |
|---|---|---|
| `[simple] ` | qwen, glm, gpt | qwen : cout declare le plus faible |
| `[analytique] ` | glm, gpt | glm : intermediaire, moins cher que gpt |
| `[raisonnement] ` | gpt seul | gpt : la politique reserve le probleme difficile a ce candidat |

Le premier connecteur enregistre est toujours candidat. Les autres n'entrent dans le tri que si `VettingLevel > None` ; la demonstration les declare `Oracle`, c'est-a-dire admis pour ce type. Le vetting en ligne de `MultiCompletionAnalysisSettings`, qui apprend ce niveau a partir d'evaluations, est hors perimetre.

Pourquoi un seul candidat en raisonnement ? Le comparateur natif calcule les rapports `cout2/cout1` et `duree2/duree1`, puis place en tete le connecteur dont les valeurs sont les plus faibles. **L'ordre d'ecriture des candidats n'est donc pas un ordre de preference de capacite.** Declarer qwen comme secours de `[raisonnement]` le ferait gagner sur le cout, au lieu de proteger la difficulte. La politique se traduit ici par l'ensemble des candidats, pas par un commentaire optimiste.

## 5. Trois appels routes, trois alias selectionnes

Le champ `selected` ne devine pas la route depuis la table : le wrapper appelle `GetMultiCompletionSession`, puis lit `session.NamedTextCompletion.Name`. La reponse, elle, passe par le `ITextGenerationService` expose par `MultiTextCompletion`.

In [6]:
var demo = RunWrapper("--demo --claudish --json");
demo.WaitForExit();
var routedLines = demo.StandardOutput.ReadToEnd()
    .Split(new[] { '\r', '\n' }, StringSplitOptions.RemoveEmptyEntries);
foreach (var line in routedLines) {
    using var document = System.Text.Json.JsonDocument.Parse(line);
    var root = document.RootElement;
    var type = root.GetProperty("type").GetString();
    var selected = root.GetProperty("selected").GetString();
    var latency = root.GetProperty("latency_ms").GetInt64();
    var text = root.GetProperty("text").GetString() ?? "";
    Console.WriteLine($"[{type}] -> {selected} | {latency} ms | {text.Length} caracteres");
    Console.WriteLine(text);
    Console.WriteLine(new string('-', 72));
}
var routingError = demo.StandardError.ReadToEnd().Trim();
Console.WriteLine($"Code retour : {demo.ExitCode}");
if (routingError.Length > 0) Console.WriteLine(routingError);

[simple] -> qwen3.6-35b-a3b | 4402 ms | 30 caracteres




At night, all cats are gray.


------------------------------------------------------------------------


[analytique] -> glm-5.3 | 5677 ms | 887 caracteres


- **Coût de gestion** : LRU (éviction de l’élément le moins récemment utilisé) est généralement simple et peu coûteux à maintenir. LFU (éviction du moins fréquemment utilisé) exige des compteurs et une organisation par fréquence, donc davantage de mémoire et de calcul.
- **Latence d’accès** : les deux peuvent offrir des opérations en temps constant avec une implémentation adaptée, mais LFU a souvent une latence légèrement supérieure à cause de la mise à jour des compteurs. En concurrence, le coût de synchronisation peut toutefois peser sur les deux.
- **Taux de succès** : LRU s’adapte vite aux changements récents, mais peut être perturbé par un parcours ponctuel de nombreuses clés. LFU protège mieux les clés durablement populaires, réduisant parfois les accès coûteux à la source, mais peut conserver trop longtemps d’anciennes clés populaires sans mécanisme de vieillissement.


------------------------------------------------------------------------


[raisonnement] -> gpt-6.1-sol | 22536 ms | 1947 caracteres


**Checking five-crossing feasibility****Refining five-crossing proof**On suppose que la barque transporte **le fermier et, au maximum, un seul des trois éléments**. Sans le fermier, le loup ne doit jamais rester avec la chèvre, ni la chèvre avec le chou.

## Plan minimal : 7 traversées

| Étape | Traversée du fermier | Rive de départ après la traversée | Rive d’arrivée après la traversée |
|---|---|---|---|
| 1 | Emmène la chèvre → | Loup, chou | Fermier, chèvre |
| 2 | Revient seul ← | Fermier, loup, chou | Chèvre |
| 3 | Emmène le loup → | Chou | Fermier, loup, chèvre |
| 4 | Ramène la chèvre ← | Fermier, chèvre, chou | Loup |
| 5 | Emmène le chou → | Chèvre | Fermier, loup, chou |
| 6 | Revient seul ← | Fermier, chèvre | Loup, chou |
| 7 | Emmène la chèvre → | Vide | Fermier, loup, chèvre, chou |

À chaque étape, les éléments laissés sans surveillance sont compatibles.

## Preuve de minimalité

Le fermier commence sur la rive de départ et termine sur l’autre : le nombre total de tra

------------------------------------------------------------------------


Code retour : 0


## Lecture du resultat

Les trois sorties sont de vraies reponses HTTP, pas des moqueurs. La route simple traduit, la route analytique compare LRU et LFU, et la route de raisonnement fournit un plan en sept traversees avec un argument de minimalite. Les durees ne sont pas des constantes : elles incluent la generation et varient avec la charge des fournisseurs.

Le resultat essentiel est la distinction entre **routage** et **qualite** : `selected` prouve que l'organe a choisi trois connecteurs differents ; il ne prouve pas qu'un modele est superieur. Une evaluation multi-instances reste necessaire avant de generaliser la politique.

## 6. Une signature inconnue ne tombe pas silencieusement sur un modele

Un prompt qui commence par `[` mais ne correspond a aucune signature retourne le code 2. Sans prefixe, le connecteur principal repond. Cette distinction rend visible une erreur de contrat au lieu de facturer une route imprevue.

In [7]:
PrintResult("Prefixe inconnu", RunWrapper("--claudish --prompt \"[inconnu] test\" --json"));

Prefixe inconnu (rc=2)


Unknown signature prefix '[inconnu]' - expected [simple], [analytique] or [raisonnement].


## Exercices

Les trois stubs ci-dessous s'executent sans erreur. Les modifications demandees portent sur `tools/multiroute/src/Program.cs`, puis sur les cellules de mesure et de demonstration.

In [8]:
// Exercice 1 — Ajouter une route [code].
// TODO etudiant : choisir deux alias de l'inventaire, les declarer dans PromptType,
// puis appeler le wrapper avec une courte fonction C# a expliquer.
// Indice : conserver VettingLevel.Oracle pour les candidats secondaires.
Console.WriteLine("Exercice a completer");

Exercice a completer


In [9]:
// Exercice 2 — Etudier la sensibilite aux profils declares.
// TODO etudiant : multiplier par quatre AverageDuration de glm dans Program.cs,
// recompiler, puis relancer les trois routes. Noter si [analytique] change d'alias.
// Etape 1 : formuler une prediction a partir du comparateur ; etape 2 : la confronter a `selected`.
Console.WriteLine("Exercice a completer");

Exercice a completer


In [10]:
// Exercice 3 — Construire une mini-evaluation avant de changer la politique.
// TODO etudiant : ecrire trois prompts [analytique] avec reponses attendues,
// les envoyer directement a glm et gpt, puis comparer exactitude et latence.
// Indice : ne pas conclure sur une seule reponse et conserver les sorties JSON.
Console.WriteLine("Exercice a completer");

Exercice a completer


## A retenir

- La fabrique `ClaudishConnectorConfiguration` transforme des alias de passerelle en connecteurs Semantic Kernel ; l'organe de routage reste inchange.
- Une route combine une **signature**, un **ensemble de candidats admis** et des **profils cout/duree**. Ces trois elements doivent etre lus ensemble.
- `VettingLevel.None` exclut les secondaires ; le comparateur natif ne signifie pas « premier = meilleur ». Ici, reserver gpt au raisonnement se fait en limitant les candidats de cette route.
- Le carnet demontre un routage reel et verifiable, pas une superiorite de modele. Sa dependance amont est explicite : `semantic-fleet#110` reste a fusionner.

## Genealogie

**Predecesseur** : [Semantic-fleet-MultiConnector-Routing](Semantic-fleet-MultiConnector-Routing.ipynb), memes organes avec OpenAI et OpenRouter directs.

**Organe natif** : `semantic-fleet/dotnet/src/Connectors/Connectors.AI.MultiConnector`, surface console `tools/multiroute`, fabrique claudish de la branche `feature/t6b-prompt-router` (`semantic-fleet#110`).

**Suivi CoursIA** : tranche t6c de #22498. Verdict SOTA : vrais appels a la passerelle et a la fabrique native ; aucun mock ni sortie de substitution.